# B0 후속 — KEJ VAD 비교 + 자유발화·대화체

셀 세 개. 코드는 `b0b_run.py` 안에 있다.

## 무엇을 재나

**A. KEJ VAD ON/OFF** — KEJ 자모 CER 0.520이 인식 실패인지, VAD가 약한 발화를 잘라낸 탓인지 가른다.
기존 `b0_16k` 를 그대로 쓴다. 새로 올릴 것 없다.

**B. 자유발화·대화체 B0** — 04 낭독은 이미 잘 된다(CTK 0.020). 낭독이 아닌 발화를 잰다.
요점은 **CTK·KEJ·KJW를 같은 화자 안에서 04 낭독과 비교**하는 것이다.

## 드라이브에 올릴 것

| 파일 | 위치 | 비고 |
| --- | --- | --- |
| `b0b_16k` 폴더 | 내 드라이브 최상위 | wav 5개 + manifest.json, **237MB** |
| `b0b_run.py` | 내 드라이브 최상위 | 새로 |
| `b0_run.py` | 내 드라이브 최상위 | **이미 있음.** CER 유틸을 여기서 가져온다 |
| `b0_16k` 폴더 | 내 드라이브 최상위 | **이미 있음** |
| `b0_results` 폴더 | 내 드라이브 최상위 | **이미 있음.** 04 낭독 값과 비교하는 데 쓴다 |

`b0_results` 가 없거나 이름이 다르면 낭독 대비 비교만 생략되고 나머지는 돈다.

**런타임 → 런타임 유형 변경 → T4 GPU → 저장.** A단계가 large-v3라 GPU가 없으면 권하지 않는다.

## 1. 설치 — `설치 OK` 가 찍혀야 다음으로 간다

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 — 런타임 유형을 T4 GPU로" ; pip -q install faster-whisper rapidfuzz && python -c "import faster_whisper, rapidfuzz; print('설치 OK')"

## 2. 드라이브 연결

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

## 3. 실행

A단계 17.2분 × 2회 + B단계 123.4분 × 2모델. **40~90분** 예상.
파일마다 줄이 쌓이고 체크포인트가 저장된다. 탭을 닫지 말 것.

In [ ]:
!python /content/drive/MyDrive/b0b_run.py

## 끝나면

`b0b_results` 에서 세 파일을 내려받아 `sw_challenge` 폴더에 넣는다.

- `b0b_summary.json`
- `kej_vad.csv`
- `b0b_per_file.csv`

### 읽는 법

**A단계** — 마지막에 판정이 한 줄로 나온다.

| VAD OFF가 | 뜻 |
| --- | --- |
| 0.02 이상 좋아짐 | VAD가 발화를 잘라먹고 있었다. KEJ 0.52는 우리 파이프라인 탓이 크다 |
| 0.02 이상 나빠짐 | VAD가 헛말을 막아주고 있었다. KEJ 0.52는 인식 실패에 가깝다 |
| 차이 없음 | VAD는 원인이 아니다 |

**B단계** — `같은 화자 안에서 낭독 대 비낭독` 표가 핵심이다. CTK가 04에서 0.020인데 03에서 크게 오르면
"낭독은 되는데 자유발화는 안 된다"가 실측으로 선다. 안 오르면 기획의 축을 다시 봐야 한다.

---

### 주의 — 03은 완전한 자유발화가 아니다

고정 질문 25개에 대한 응답이고 템플릿이 있다.

> 제 나이는 O살입니다. 저는 O띠입니다. 제가 태어난 곳은 O이고...

지명·가족 구성·숫자 같은 내용어는 화자마다 달라서 대본 낭독보다 예측이 어렵지만, **진짜 자유발화는 아니다.**
결과를 "자유발화 성능"이라고 단정하지 말고 "준자유발화"로 쓴다. 이 데이터에 완전 자유발화 과제는 없다.

### 안 될 때

| 증상 | 조치 |
| --- | --- |
| `b0_run.py 를 못 찾았다` | `b0_run.py`를 드라이브 최상위에 두기 |
| `b0b_16k/manifest.json 이 없다` | `b0b_16k` 폴더를 최상위로 |
| `이전 결과 ... 없음` | 경고일 뿐. 낭독 대비 비교만 빠진다 |
| 런타임 끊김 | 다시 연결 후 1번부터. **자동 재개 없음** |